<a href="https://colab.research.google.com/github/dtoralg/TheValley_MDS/blob/main/%5B03%5D%20-%20Algoritmos_Alternativos_Clasificacion/%5B01%5D%20-%20Notebooks/E6_SVM_Breast_Cancer_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clasificación de Cáncer de Mama con SVM

En este ejercicio utilizamos máquinas de vectores de soporte (SVM) para predecir si un tumor es maligno o benigno a partir de características obtenidas por imágenes médicas.

## Objetivos:
- Comprender cómo funciona SVM y sus variantes (kernels).
- Explorar cómo afectan los hiperparámetros `C`, `gamma` y el tipo de kernel.
- Comparar rendimiento y capacidad de generalización.
- Visualizar fronteras de decisión y márgenes en 2D.

## Cargar y explorar los datos

In [ ]:
from sklearn.datasets import load_breast_cancer
import pandas as pd

data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

df['target'].value_counts()
df.describ

## 📌 Visualización con PCA

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

pca = PCA(n_components=2)
X_pca = pca.fit_transform(data.data)

plt.figure(figsize=(8,6))
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=data.target, palette="Set2")
plt.title("Visualización PCA - Cáncer de mama")
plt.xlabel("Componente 1")
plt.ylabel("Componente 2")
plt.show()

## 📌 Preprocesamiento y división train/test

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## 📌 Entrenamiento con SVM lineal

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

svc_linear = SVC(kernel='linear', C=1)
svc_linear.fit(X_train, y_train)
y_pred_lin = svc_linear.predict(X_test)

print("SVM Lineal")
print(classification_report(y_test, y_pred_lin))
sns.heatmap(confusion_matrix(y_test, y_pred_lin), annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix - SVM Lineal")
plt.show()

## 📌 Comparación con kernel RBF

In [ ]:
svc_rbf = SVC(kernel='rbf', gamma='scale', C=1)
svc_rbf.fit(X_train, y_train)
y_pred_rbf = svc_rbf.predict(X_test)

print("SVM RBF")
print(classification_report(y_test, y_pred_rbf))
sns.heatmap(confusion_matrix(y_test, y_pred_rbf), annot=True, fmt='d', cmap='Greens')
plt.title("Confusion Matrix - SVM RBF")
plt.show()

## 📌 Comparación con kernel polinómico

In [ ]:
svc_poly = SVC(kernel='poly', degree=3, C=1)
svc_poly.fit(X_train, y_train)
y_pred_poly = svc_poly.predict(X_test)

print("SVM Polinómico")
print(classification_report(y_test, y_pred_poly))
sns.heatmap(confusion_matrix(y_test, y_pred_poly), annot=True, fmt='d', cmap='Oranges')
plt.title("Confusion Matrix - SVM Polinómico")
plt.show()

## 📌 Grid Search para ajuste de hiperparámetros

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 0.01, 0.001],
    'kernel': ['rbf']
}

grid = GridSearchCV(SVC(), param_grid, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

print("Mejores parámetros:", grid.best_params_)
print("Mejor score (cross-val):", grid.best_score_)

## 📌 Visualización de fronteras con 2 features

In [ ]:
# Usar solo 2 variables para visualizar frontera
from mlxtend.plotting import plot_decision_regions

X_vis = X_train[:, :2]  # solo dos primeras features
svm_vis = SVC(kernel='linear', C=1)
svm_vis.fit(X_vis, y_train)

plot_decision_regions(X_vis, y_train, clf=svm_vis, legend=2)
plt.xlabel(data.feature_names[0])
plt.ylabel(data.feature_names[1])
plt.title('Frontera de decisión - SVM Lineal (2 features)')
plt.show()

## 📌 Reflexión final

## Reflexión

- ¿Qué kernel ha funcionado mejor? ¿Por qué?
- ¿Cómo influye el parámetro `C` (tolerancia al error)? ¿Y `gamma`?
- ¿Qué pasa si las clases no son linealmente separables?
- ¿Qué ventaja tiene SVM respecto a un árbol de decisión o regresión logística?

## Recomendaciones

- Usa SVM cuando:
  - Hay pocos outliers y los datos son numéricos.
  - El número de muestras no es gigantesco.
  - Buscas buena capacidad de generalización.

- Evita SVM cuando:
  - El dataset es muy grande (es costoso computacionalmente).
  - Hay mucho ruido o las clases se solapan mucho.

SVM es un modelo **potente y elegante**, que bien parametrizado puede ofrecer excelentes resultados. Pero también es sensible a la escala y requiere cierto cuidado en su ajuste.